In [1]:
from datetime import date, timedelta
import random
from faker import Faker


def generate_customers(count: int = 1000, seed: int = 42) -> list[dict]:
    fake = Faker("id_ID")
    Faker.seed(seed)

    cities = [
        "Jakarta",
        "Bandung",
        "Surabaya",
        "Semarang",
        "Yogyakarta",
        "Medan",
        "Makassar",
        "Denpasar",
        "Bekasi",
        "Tangerang",
    ]

    customers = []

    start_date = date(2020, 1, 1)
    end_date = date(2025, 12, 31)
    date_range = (end_date - start_date).days

    for i in range(1, count + 1):
        signup_date = start_date + timedelta(
            days=fake.random_int(0, date_range)
        )

        cust_name = fake.name()

        customers.append({
            "customer_id": f"C{i:06d}",
            "name": cust_name,
            "email": f"customer{cust_name.lower().replace(' ', '')}@example.com",
            "city": fake.random_element(cities),
            "signup_date": signup_date,
        })

    return customers

customers = generate_customers()
print(customers[random.randint(1, 1000)])

{'customer_id': 'C000858', 'name': 'Timbul Marbun, S.IP', 'email': 'customertimbulmarbun,s.ip@example.com', 'city': 'Surabaya', 'signup_date': datetime.date(2024, 9, 23)}


In [2]:
from faker import Faker
import faker_commerce


def generate_products(count: int = 1000, seed: int = 42) -> list[dict]:
    fake = Faker("id_ID")
    Faker.seed(seed)
    fake.add_provider(faker_commerce.Provider)

    products = []

    for i in range(1, count + 1):

        price = int(
            fake.random_int(
                min=1,
                max=500
            ) * 1000
        )


        products.append({
            "product_id": f"P{i:06d}",
            "product_name": f"{fake.ecommerce_name()} - {i:06d}",
            "category": fake.ecommerce_category(),
            "price": price,
        })

    return products

products = generate_products()
print(products[random.randint(1, 1000)])

{'product_id': 'P000383', 'product_name': 'Gorgeous Granite Cheese - 000383', 'category': 'Grocery', 'price': 42000}


In [ ]:
from datetime import date, timedelta
from faker import Faker


def generate_marketing_campaigns(
    count: int = 1000,
    seed: int = 42
) -> list[dict]:

    fake = Faker()
    Faker.seed(seed)

    channels = [
        "Email",
        "Social Media",
        "SMS",
        "Push Notification",
        "Affiliate",
        "Display Ads",
    ]

    campaign_names = [
        "New Year Sale",
        "Ramadan Sale",
        "Lebaran Sale",
        "Back to School",
        "Independence Day",
        "Black Friday",
        "Year End Sale",
        "Flash Sale",
        "Weekend Sale",
        "Member Exclusive",
    ]

    campaigns = []
    base_date = date(2020, 1, 1)

    for i in range(1, count + 1):
        start_date = base_date + timedelta(
            days=fake.random_int(0, 2000)
        )

        duration = fake.random_int(3, 30)
        end_date = start_date + timedelta(days=duration)

        campaigns.append({
            "campaign_id": f"CAM{i:06d}",
            "campaign_name": (
                f"{fake.random_element(campaign_names)} {i:06d}"
            ),
            "start_date": start_date,
            "end_date": end_date,
            "channel": fake.random_element(channels),
        })

    return campaigns

campaigns = generate_marketing_campaigns()
print(campaigns[random.randint(1, 1000)])

{'campaign_id': 'CAM000071', 'campaign_name': 'Year End Sale 000071', 'start_date': datetime.date(2022, 6, 1), 'end_date': datetime.date(2022, 6, 15), 'channel': 'Push Notification'}


In [6]:
from datetime import date, timedelta
from faker import Faker


def generate_transactions(
    customers: list[dict],
    count: int = 1000,
    seed: int = 42
) -> list[dict]:

    fake = Faker()
    Faker.seed(seed)

    transactions = []

    start_date = date(2023, 1, 1)
    end_date = date(2025, 12, 31)

    date_range = (end_date - start_date).days

    for i in range(1, count + 1):

        customer = fake.random_element(customers)

        transaction_date = start_date + timedelta(
            days=fake.random_int(0, date_range)
        )

        transactions.append({
            "transaction_id": f"T{i:06d}",
            "customer_id": customer["customer_id"],
            "transaction_date": transaction_date,

            # Akan dihitung setelah transaction_items dibuat
            "total_amount": 0,
        })

    return transactions

transactions = generate_transactions(customers)
print(transactions[random.randint(1, 1000)])

{'transaction_id': 'T000055', 'customer_id': 'C000218', 'transaction_date': datetime.date(2025, 10, 19), 'total_amount': 0}


In [7]:
from faker import Faker


def generate_transaction_items(
    transactions: list[dict],
    products: list[dict],
    min_items: int = 1,
    max_items: int = 5,
    seed: int = 42
) -> list[dict]:

    fake = Faker()
    Faker.seed(seed)

    transaction_items = []

    item_id = 1

    for transaction in transactions:

        item_count = fake.random_int(
            min=min_items,
            max=max_items
        )

        selected_products = fake.random_elements(
            elements=products,
            length=item_count,
            unique=True
        )

        transaction_total = 0

        for product in selected_products:

            quantity = fake.random_int(
                min=1,
                max=5
            )

            price = product["price"]

            amount = quantity * price

            transaction_items.append({
                "transaction_item_id": f"TI{item_id:07d}",
                "transaction_id": transaction["transaction_id"],
                "product_id": product["product_id"],
                "quantity": quantity,
                "price": price,
            })

            transaction_total += amount
            item_id += 1

        transaction["total_amount"] = transaction_total

    return transaction_items

transaction_items = generate_transaction_items(transactions, products)
print(transactions[random.randint(1, 1000)])

{'transaction_id': 'T000985', 'customer_id': 'C000366', 'transaction_date': datetime.date(2023, 5, 19), 'total_amount': 1866000}


In [8]:
import csv
from pathlib import Path

OUTPUT_DIR = Path("data/sample")
ROW_COUNT = 1000
SEED = 42


def write_csv(
    filename: str,
    rows: list[dict]
):
    if not rows:
        return

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    filepath = OUTPUT_DIR / filename

    with filepath.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=rows[0].keys()
        )

        writer.writeheader()
        writer.writerows(rows)

    print(f"Generated {filepath} ({len(rows)} rows)")

In [10]:
# 1. Master data
customers = generate_customers(count=ROW_COUNT, seed=SEED)
products = generate_products(count=ROW_COUNT, seed=SEED)
campaigns = generate_marketing_campaigns(count=ROW_COUNT, seed=SEED)

# 2. Transaction header
transactions = generate_transactions(customers=customers, count=ROW_COUNT, seed=SEED)

# 3. Transaction details
transaction_items = generate_transaction_items(transactions=transactions, products=products, seed=SEED)

# 4. Write source files
write_csv("customers.csv", customers)
write_csv("products.csv", products)
write_csv("transactions.csv", transactions)
write_csv("transaction_items.csv", transaction_items)
write_csv("marketing_campaigns.csv", campaigns)

Generated data/sample/customers.csv (1000 rows)
Generated data/sample/products.csv (1000 rows)
Generated data/sample/transactions.csv (1000 rows)
Generated data/sample/transaction_items.csv (3014 rows)
Generated data/sample/marketing_campaigns.csv (1000 rows)
